# Robustness check — rolling temporal splits (SPX)

Same design as the AAPL robustness notebook: re-run the main pipeline on different
temporal splits of the *same, already-downloaded* dataset, to check whether the central
result (Black--Scholes winning in the money, XGBoost winning out of the money) is
specific to the original split or holds more generally.

- Rolling, non-overlapping windows (same training length across splits, so a difference
  across splits reflects *which* period, not *how much* data).
- Same five inputs, same v4 target (`log(C/K)`) as the main model.
- Same hyperparameters already selected via rolling-window CV on the original SPX split
  (Table 10) — no new grid search.
- XGBoost only.
- **Before running**: confirm `SPX_cleaned.csv` covers the full sample used in the
  thesis (7,688,150 rows, 31 Aug 2020 to 29 Aug 2025) — the AAPL version of this
  robustness check first ran on a truncated file by mistake, producing empty splits.
  Run the check in the first code cell below before proceeding to the rest.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

import joblib


In [ ]:
spx = pd.read_csv("SPX_cleaned.csv")
spx["date"] = pd.to_datetime(spx["date"])
spx["exdate"] = pd.to_datetime(spx["exdate"])

print(spx.shape)
print(spx["date"].min(), spx["date"].max())
spx.head()


(7688150, 17)
2020-08-31 00:00:00 2025-08-29 00:00:00


,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price,volatility,rate,days_to_maturity,T,q
0,108105,2020-08-31,2020-09-18,100.0,3402.5,3408.2,0,611,133485146,3500.31,35.003100,3405.35,0.111899,0.002148,18,0.049315,0.01589
1,108105,2020-08-31,2020-09-18,1000.0,2503.2,2508.0,0,37663,130915017,3500.31,3.500310,2505.60,0.111899,0.002148,18,0.049315,0.01589
2,108105,2020-08-31,2020-09-18,1100.0,2403.4,2407.9,0,96,130915018,3500.31,3.182100,2405.65,0.111899,0.002148,18,0.049315,0.01589
3,108105,2020-08-31,2020-09-18,1200.0,2303.5,2307.9,0,19,129372797,3500.31,2.916925,2305.70,0.111899,0.002148,18,0.049315,0.01589
4,108105,2020-08-31,2020-09-18,1250.0,2253.7,2258.0,0,20,129372798,3500.31,2.800248,2255.85,0.111899,0.002148,18,0.049315,0.01589


In [ ]:
# Sanity check before doing anything else: this must print (7688150, ...) and
# 2020-08-31 ... 2025-08-29. If it doesn't, stop and fix the input file first.
assert spx.shape[0] > 7_000_000, f"Unexpected row count: {spx.shape[0]:,} — wrong file?"
assert spx["date"].max().year == 2025, f"Unexpected max date: {spx['date'].max()} — wrong file?"
print("OK — full SPX sample loaded.")


OK — full SPX sample loaded.


## Shared settings (identical to the main notebook)

In [ ]:
BINS   = [0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100]
LABELS = ["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]

def bucket_mae(df, preds):
    """MAE by moneyness bucket for BS and each model; last row = all options."""
    t = pd.DataFrame({"bucket": pd.cut(df["moneyness (S/K)"], bins=BINS, labels=LABELS),
                      "BS": np.abs(df["price"].values - df["bs_price"].values)})
    for k, p in preds.items():
        t[k] = np.abs(df["price"].values - np.asarray(p))
    out = t.groupby("bucket", observed=True).mean()
    out.insert(0, "n", t.groupby("bucket", observed=True).size())
    out.loc["ALL"] = [len(t)] + list(t.drop(columns="bucket").mean())
    return out.astype({"n": int})

cv_feature_cols = ["moneyness (S/K)", "T", "rate", "q", "volatility"]
cv_target_col = "price"


## 1. Black-Scholes benchmark

Identical formula and inputs as the main notebook. BS price does not depend on the
split, so it is computed once on the full dataset.


In [ ]:
d1 = (
    np.log(spx["close"] / spx["strike_price"])
    + (spx["rate"] - spx["q"] + 0.5 * spx["volatility"] ** 2) * spx["T"]
) / (spx["volatility"] * np.sqrt(spx["T"]))

d2 = d1 - spx["volatility"] * np.sqrt(spx["T"])

spx["bs_price"] = (
    spx["close"] * np.exp(-spx["q"] * spx["T"]) * norm.cdf(d1)
    - spx["strike_price"] * np.exp(-spx["rate"] * spx["T"]) * norm.cdf(d2)
)

spx["bs_price"].describe()


,bs_price
count,7.688150e+06
mean,4.168183e+02
std,6.769914e+02
min,0.000000e+00
25%,3.834214e+01
50%,1.692638e+02
75%,4.849242e+02
max,6.298850e+03


## 2. Reused hyperparameters (from Table 10, main analysis)

No new grid search: the goal is to test robustness of the *result*, not to re-tune the
model for each split.


In [ ]:
best_params = {"max_depth": 5, "learning_rate": 0.05, "n_estimators": 100, "subsample": 0.8}
best_params = {k: (int(v) if k in ["max_depth", "n_estimators"] else float(v)) for k, v in best_params.items()}
print(best_params)


{'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 100, 'subsample': 0.8}


## 3. Three rolling, non-overlapping splits

Same date boundaries used for the AAPL robustness check, for direct comparability
across the two underlyings. Adjust here if AAPL's boundaries changed after checking
data coverage.


In [ ]:
splits = {
    "C": {
        "train": ("2020-08-31", "2021-08-31"),
        "val":   ("2021-09-01", "2022-01-01"),
        "test":  ("2022-01-02", "2022-04-30"),
    },
    "B": {
        "train": ("2022-05-01", "2023-05-01"),
        "val":   ("2023-05-02", "2023-09-01"),
        "test":  ("2023-09-02", "2023-12-31"),
    },
    "A": {
        "train": ("2024-01-01", "2025-01-01"),
        "val":   ("2025-01-02", "2025-05-01"),
        "test":  ("2025-05-02", "2025-08-29"),
    },
}

for name, s in splits.items():
    print(name, s)


C {'train': ('2020-08-31', '2021-08-31'), 'val': ('2021-09-01', '2022-01-01'), 'test': ('2022-01-02', '2022-04-30')}
B {'train': ('2022-05-01', '2023-05-01'), 'val': ('2023-05-02', '2023-09-01'), 'test': ('2023-09-02', '2023-12-31')}
A {'train': ('2024-01-01', '2025-01-01'), 'val': ('2025-01-02', '2025-05-01'), 'test': ('2025-05-02', '2025-08-29')}


## 4. Fit and evaluate each split

Same recipe as the main notebook's final model: fit on train+val with `log(C/K)` as
target, evaluate on test, convert predictions back with `exp(pred) * K`.


In [ ]:
def fit_eval_split(df, split_dates, label):
    tr = df[(df["date"] >= split_dates["train"][0]) & (df["date"] <= split_dates["train"][1])]
    va = df[(df["date"] >= split_dates["val"][0])   & (df["date"] <= split_dates["val"][1])]
    te = df[(df["date"] >= split_dates["test"][0])  & (df["date"] <= split_dates["test"][1])]
    tv = pd.concat([tr, va]).sort_values("date").reset_index(drop=True)
    te = te.sort_values("date").reset_index(drop=True)

    print(f"[{label}] train {len(tr):,} | val {len(va):,} | train+val {len(tv):,} | test {len(te):,}")
    print(f"[{label}] test window: {te['date'].min()} to {te['date'].max()}")

    X_tv = tv[cv_feature_cols]
    y_tv_log = np.log(tv[cv_target_col] / tv["strike_price"])

    model = xgb.XGBRegressor(**best_params, tree_method="hist", random_state=42, n_jobs=-1)
    model.fit(X_tv, y_tv_log)

    X_te = te[cv_feature_cols]
    test_pred = np.exp(model.predict(X_te)) * te["strike_price"].values

    return te, test_pred, model

results = {}
for name, dates in splits.items():
    te, pred, model = fit_eval_split(spx, dates, f"SPX-{name}")
    results[name] = {"test": te, "pred": pred, "model": model}


[SPX-C] train 1,229,480 | val 476,140 | train+val 1,705,620 | test 481,213
[SPX-C] test window: 2022-01-03 00:00:00 to 2022-04-29 00:00:00
[SPX-B] train 1,455,378 | val 519,178 | train+val 1,974,556 | test 507,984
[SPX-B] test window: 2023-09-05 00:00:00 to 2023-12-29 00:00:00
[SPX-A] train 1,762,789 | val 625,038 | train+val 2,387,827 | test 630,950
[SPX-A] test window: 2025-05-02 00:00:00 to 2025-08-29 00:00:00


## 5. Per-bucket MAE, each split

In [ ]:
bucket_tables = {}
for name, r in results.items():
    bucket_tables[name] = bucket_mae(r["test"], {"XGB": r["pred"]})
    print(f"--- Split {name} ---")
    print(bucket_tables[name])
    print()


--- Split C ---
                    n         BS          XGB
bucket                                       
Deep OTM        15912   7.109710     1.020079
OTM            132956  15.045603     9.248130
ATM            177751  19.556221    18.821665
ITM            100939  29.135890    16.464970
Deep ITM        40252  28.615935    32.055407
Very Deep ITM   10072   7.474022   164.615459
Extreme ITM      3331   5.795984  1176.532243
ALL            481213  20.317513    27.265854

--- Split B ---
                    n         BS          XGB
bucket                                       
Deep OTM         8527   0.776898     0.312009
OTM             89300   5.968536     4.539328
ATM            224678  14.321074    10.394234
ITM            117554  18.658930    11.954758
Deep ITM        50532  12.473524    54.986109
Very Deep ITM   12013   2.799430   221.709759
Extreme ITM      5380   2.392322  1092.294481
ALL            507984  13.046650    30.448222

--- Split A ---
                    n         

## 6. Side-by-side comparison


In [ ]:
summary = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    summary[f"BS_{name}"] = t["BS"]
    summary[f"XGB_{name}"] = t["XGB"]

summary.round(3)


,BS_C,XGB_C,BS_B,XGB_B,BS_A,XGB_A
Deep OTM,7.110,1.020,0.777,0.312,45.055,0.635
OTM,15.046,9.248,5.969,4.539,76.943,9.118
ATM,19.556,18.822,14.321,10.394,57.170,21.263
ITM,29.136,16.465,18.659,11.955,46.292,13.948
Deep ITM,28.616,32.055,12.474,54.986,17.195,36.507
Very Deep ITM,7.474,164.615,2.799,221.710,4.167,363.366
Extreme ITM,5.796,1176.532,2.392,1092.294,3.554,1505.523
ALL,20.318,27.266,13.047,30.448,50.000,36.190


In [ ]:
# Same comparison expressed as a ratio (XGB error / BS error): <1 means XGB wins
ratio = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    ratio[name] = t["XGB"] / t["BS"]

ratio.round(3)


,C,B,A
Deep OTM,0.143,0.402,0.014
OTM,0.615,0.761,0.119
ATM,0.962,0.726,0.372
ITM,0.565,0.641,0.301
Deep ITM,1.120,4.408,2.123
Very Deep ITM,22.025,79.198,87.207
Extreme ITM,202.991,456.583,423.620
ALL,1.342,2.334,0.724


## 7. Volatility check for split C

Verifies whether split C's test window (Jan-Apr 2022) is indeed a higher-volatility
period, as a candidate explanation if the out-of-the-money advantage reverses there —
consistent with the smile-based mechanism proposed in Section 2.3.


In [ ]:
for name, s in splits.items():
    window = spx[(spx["date"] >= s["test"][0]) & (spx["date"] <= s["test"][1])]
    print(f"Split {name} test window volatility — mean: {window['volatility'].mean():.4f}, "
          f"median: {window['volatility'].median():.4f}")


Split C test window volatility — mean: 0.2012, median: 0.2055
Split B test window volatility — mean: 0.1236, median: 0.1238
Split A test window volatility — mean: 0.1903, median: 0.1338


## 8. Save results

In [ ]:
joblib.dump(
    {"splits": splits, "best_params": best_params, "summary": summary, "ratio": ratio,
     "bucket_tables": bucket_tables},
    "robustness_spx_results.pkl"
)


['robustness_spx_results.pkl']